In [11]:
import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
import time
import re

liste_competences = [
    "solidworks", "catia", "autocad", "creo", "nx", "pro/engineer","inventor", 
    "ansys", "matlab", "simulink", "abaqus", "samcef","nastran", "patran", "icem", 
    "cfd", "fea", "rdm", "cao", "dao", "conception mécanique", "modélisation",
    "fabrication additive", "gmao", "sap", "erp", "ms project", "automatisme", 
    "robotique", "amdec", "apqp", "ppap", "spc", "5s", "kaizen", "lean", 
    "lean manufacturing", "six sigma", "python", "sql", "vba", "excel"
]
liste_soft_skills = [
    "communication", "communiquer", "esprit d'équipe", "travail en équipe", "travailler en équipe",
    "autonomie","autonome", "rigueur", "organisation", "organisé", "créativité", "créatif",
    "leadership", "gestion de projet","innovation", "résolution de problèmes", "adaptabilité",
    "force de proposition", "proactivité", "relationnel", "initiative", "polyvalence", "flexibilité", 
    "sens de l'organisation", "réactivité", "esprit d'analyse"
]

scraper = cloudscraper.create_scraper()                           #contourner code 403
liste_offres = []

for page in range(1, 5):                                               # pour parcourir les pages 
    url = "https://www.rekrute.com/fr/offres.html?s=1&p=" + str(page) + "&o=1&query=" + "genie+mecanique" + "&keyword=" + "genie+mecanique"
    try:
        response = scraper.get(url, timeout=15)                        #envoie des requetes au site
    except Exception as e:
        print("Erreur lors du scraping de la page", page, ":", e)
        continue

    if response.status_code != 200:
        print("Erreur de connexion sur la page", page, "(Code:", response.status_code, ")")
        continue

    soup = BeautifulSoup(response.text, "html.parser")                     #lecture  du code html(parsing)
    offres_html = soup.select("li.post-id")                                #extraxtion du texte de la balise li classe postid
    compteur_page = 0

    for offre_html in offres_html:                                         # extraction du titre et du lien de chaque offre

        titre_tag = offre_html.select_one("a.titreJob")
        if titre_tag is None:
            continue
                                                                 
        titre_complet = titre_tag.text.strip()                           #nettoyage du titre 
        lien = titre_tag.get("href", "")

        lien_complet = "https://www.rekrute.com" + lien
        response = scraper.get(lien_complet, timeout=15)                            #requetes pour parcouri chaque offre(page detaillee)
        soup = BeautifulSoup(response.text, "html.parser")
        texte = soup.get_text(" ", strip=True).lower() 
                                                                                          #extraction des titres et des villes 
        if "|" in titre_complet:
            morceaux = titre_complet.split("|")
            titre = morceaux[0].strip()
            ville = morceaux[1].strip()
            ville = re.sub(r"\s*\(.*?\)", "", ville)   
            ville = ville.strip() 
        else:
            titre = titre_complet
            ville = ""
                                                                                   
        img_tag = offre_html.select_one(".col-sm-2 img.photo")                            #extraction du nom de l entreprise via son logo
        if img_tag is not None:
            entreprise = img_tag.get("title", "Confidentiel")  
        else:
            entreprise = "Confidentiel"

                                                                   #extraction des langues requises
        langues = []
        for l in ["français", "francais", "anglais", "english", "arabe"]:                    #recherche des langues dans le texte
            if l in texte:
                langues.append(l.capitalize())
        langue = ", ".join(langues) if langues else "Non spécifié"
 
        competences = []
        for c in liste_competences:
            if c in texte:
                if c in ["solidworks", "catia", "nx", "ansys", 
                         "matlab", "samcef", "nastran", "icem", 
                         "cfd", "fea", "rdm", "cao", "dao", 
                         "gmao", "sap", "erp", "amdec", "apqp", 
                         "ppap", "spc", "5s", "sql", "vba"]: 
                    competences.append(c.upper())
                else:
                    competences.append(c.capitalize())
        competence = ", ".join(competences) if competences else "Non spécifié"  
        
        soft_skills = []
        for s in liste_soft_skills:
            if s in texte:
                soft_skills.append(s)
        soft_skill = ", ".join(soft_skills).capitalize() if soft_skills else "Non spécifié"  
        
        contrat = "Non spécifié"
        niveau_etude = "Non spécifié"                                                     #initialisation et extraction de ces 3 colonnes par la balise ul.featureInfo li
        experience = "Non spécifié"
        secteur = "Non spécifié"

        for li in offre_html.select("div.info li"):
            texte = li.get_text(" ", strip=True)

            if "Expérience requise" in texte:
                a = li.find("a")
                experience = a.get_text(strip=True) if a else texte.replace("Expérience requise :", "").strip()

            elif "Niveau d'étude demandé" in texte:
                a = li.find("a")
                niveau_etude = a.get_text(strip=True) if a else texte.replace("Niveau d'étude demandé :", "").strip()

            elif "Type de contrat" in texte:
                a = li.find("a")
                contrat = a.get_text(strip=True) if a else texte.replace("Type de contrat :", "").strip()

            elif "Secteur d'activité" in texte:
                a = li.find("a")
                secteur = a.get_text(strip=True) if a else texte.replace("Secteur d'activité :", "").strip()
                    
#stockage des informations dans un dic
        offre = {
            "Titre du poste": titre,
            "Entreprise": entreprise,
            "Ville": ville,
            "Secteur d'activité" : secteur,
            "Contrat proposé": contrat,  
            "Niveau d'étude": niveau_etude,
            "Expérience": experience, 
            "Compétences techniques": competence,
            "Langues": langue,
            "Soft skills": soft_skill,
            "Lien de l'offre": "https://www.rekrute.com" + lien
        }
#trasnfert dans une liste
        liste_offres.append(offre)
        compteur_page = compteur_page + 1
    time.sleep(1.5)

if len(liste_offres) > 0:
    df_rekrute = pd.DataFrame(liste_offres)
    df_rekrute.to_csv("offres_rekrute_mecanique.csv", index=False, sep=",", encoding="utf-8-sig")
    print("Extraction terminee. Total :", len(df_rekrute), "offres")
else:
    print("Aucune offre")

Extraction terminee. Total : 34 offres


In [13]:
df = pd.read_csv("offres_rekrute_mecanique.csv", sep=",")
df["Niveau d'étude"] = df["Niveau d'étude"].str.replace("et plus", " ", regex=False)
df = df.replace("Autre", "Non spécifié")

df.to_csv("offres_rekrute_mecanique.csv", index=False, sep=",", encoding="utf-8-sig")

df.head(20)

,Titre du poste,Entreprise,Ville,Secteur d'activité,Contrat proposé,Niveau d'étude,Expérience,Compétences techniques,Langues,Soft skills,Lien de l'offre
0,Responsable SAV,Auto Hall,Casablanca,Automobile / Motos / Cycles,CDI,Bac +5,Junior (1 à 3 ans),Excel,Non spécifié,"Travail en équipe, innovation, initiative, fle...",https://www.rekrute.com/fr/offre-emploi-respon...
1,Ingénieur Installation Générale Senior,Capgemini Engineering,Casablanca,BTP / Génie Civil,CDI,Bac +5,Intermédiaire (3 à 5 ans),ERP,Anglais,"Communication, travailler en équipe, autonomie...",https://www.rekrute.com/fr/offre-emploi-ingeni...
2,Directeur Technique,Confidentiel,Marrakech,Hôtellerie / Restauration,CDI,Bac +5,Confirmé (5 à 10 ans),Non spécifié,"Français, Anglais, Arabe","Travail en équipe, rigueur, organisation, lead...",https://www.rekrute.com/fr/offre-emploi-direct...
3,Responsable Maintenance,Sedec,Témara,BTP / Génie Civil,CDI,Bac +5,Confirmé (5 à 10 ans),"GMAO, Automatisme",Non spécifié,"Organisation, leadership, gestion de projet, i...",https://www.rekrute.com/fr/offre-emploi-respon...
4,Facility Manager,Manpower Agences,Casablanca,Autres Industries,Intérim,Bac +5,Confirmé (5 à 10 ans),Excel,"Français, Anglais","Communication, rigueur, organisation, leadersh...",https://www.rekrute.com/fr/offre-emploi-facili...
5,Technicien centrale chaufferie,Confidentiel,Casablanca,Agroalimentaire,Non spécifié,Bac +2,Débutant (-1 an),Automatisme,Non spécifié,Non spécifié,https://www.rekrute.com/fr/offre-emploi-techni...
6,Préparateur Maintenance – Maintenance Fixe,Aya Gold & Silver,Taliouine- Askaoune,Extraction / Mines,CDI,Bac +2,Intermédiaire (3 à 5 ans),"GMAO, ERP, Excel","Français, Anglais","Rigueur, organisation, relationnel",https://www.rekrute.com/fr/offre-emploi-prepar...
7,Technicien(ne) supérieur(e) - Secteur Hydrauli...,Manpower Agences,Casablanca,Energie,Non spécifié,Bac +3,Junior (1 à 3 ans),"ERP, Excel","Français, Anglais","Rigueur, organisation, innovation, adaptabilité",https://www.rekrute.com/fr/offre-emploi-techni...
8,TECHNICIEN DE FABRICATION,Cosumar,Kénitra,Agroalimentaire,Non spécifié,Bac +3,Débutant (-1 an),Non spécifié,Non spécifié,"Travail en équipe, rigueur, organisation, inno...",https://www.rekrute.com/fr/offre-emploi-techni...
9,Dessinateur projeteur,Confidentiel,Casablanca,Conseil / Etudes,CDI,Bac +2,Junior (1 à 3 ans),ERP,Non spécifié,"Communication, travail en équipe, autonomie, r...",https://www.rekrute.com/fr/offre-emploi-dessin...
